## Example 1: Public API and configuration context

In [ ]:
import pandas_gbq

# Example 1: public entry points and the module-level configuration context.
# Nothing here talks to BigQuery or needs credentials.
print(f"pandas_gbq version: {getattr(pandas_gbq, '__version__', 'unknown')}")

for name in ('read_gbq', 'to_gbq'):
    assert callable(getattr(pandas_gbq, name)), f"pandas_gbq.{name} is missing"

ctx = pandas_gbq.context
saved = (ctx.dialect, ctx.project)
try:
    ctx.dialect = 'standard'
    ctx.project = 'example-project'
    print(f"context.dialect={ctx.dialect!r}, context.project={ctx.project!r}")
    assert ctx.dialect == 'standard'
    assert ctx.project == 'example-project'
finally:
    ctx.dialect, ctx.project = saved

print("Public API and context checks completed")

## Example 2: read_gbq argument validation

In [ ]:
import pandas_gbq

# Example 2: read_gbq validates its arguments before it builds a BigQuery
# client, so an invalid SQL dialect fails without credentials or network.
try:
    pandas_gbq.read_gbq("SELECT 1 AS x", project_id="example-project", dialect="bogus")
except ValueError as e:
    print(f"read_gbq rejected the dialect: {e}")
    assert "dialect" in str(e)
else:
    raise AssertionError("read_gbq accepted an invalid dialect")

print("read_gbq argument validation completed")

## Example 3: Schema generation from a DataFrame

In [ ]:
import pandas as pd
import pandas_gbq

# Example 3: the BigQuery schema pandas_gbq generates for a DataFrame upload.
# Newer releases keep the generator in pandas_gbq.gbq; older ones exposed it in pandas_gbq.schema.
generate_bq_schema = (getattr(pandas_gbq.schema, 'generate_bq_schema', None)
                      or pandas_gbq.gbq._generate_bq_schema)

df = pd.DataFrame({
    'user_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'salary': [50000.50, 65000.00, 45000.75],
    'is_active': [True, True, False],
    'updated_at': pd.to_datetime(['2020-01-15', '2019-08-22', '2021-03-10']).tz_localize('UTC'),
})
print(f"DataFrame dtypes:\n{df.dtypes}")

schema = generate_bq_schema(df)
types = {field['name']: field['type'] for field in schema['fields']}
print(f"Generated schema types: {types}")

assert [field['name'] for field in schema['fields']] == list(df.columns)
assert types['user_id'] == 'INTEGER'
assert types['name'] == 'STRING'
assert types['salary'] == 'FLOAT'
assert types['is_active'] == 'BOOLEAN'
assert types['updated_at'] == 'TIMESTAMP'

print("Schema generation completed")

## Example 4: Schema helpers

In [ ]:
from pandas_gbq import schema as gbq_schema

# Example 4: helpers that merge and compare BigQuery schemas
local = {'fields': [
    {'name': 'id', 'type': 'INTEGER'},
    {'name': 'payload', 'type': 'STRING'},
    {'name': 'score', 'type': 'FLOAT'},
]}

# update_schema overrides the type of a matching field and keeps the others
updated = gbq_schema.update_schema(local, {'fields': [{'name': 'payload', 'type': 'BYTES'}]})
updated_types = {field['name']: field['type'] for field in updated['fields']}
print(f"Updated schema types: {updated_types}")
assert updated_types == {'id': 'INTEGER', 'payload': 'BYTES', 'score': 'FLOAT'}

# schema_is_subset: every local field must exist in the remote schema
remote = {'fields': local['fields'] + [{'name': 'extra', 'type': 'STRING'}]}
assert gbq_schema.schema_is_subset(remote, local)
assert not gbq_schema.schema_is_subset(local, remote)

# Conversion to google-cloud-bigquery SchemaField objects
fields = gbq_schema.to_google_cloud_bigquery(local)
print(f"SchemaField names: {[f.name for f in fields]}")
assert [f.name for f in fields] == ['id', 'payload', 'score']
assert [f.field_type for f in fields] == ['INTEGER', 'STRING', 'FLOAT']

print("Schema helpers completed")

## Example 5: to_gbq argument validation

In [ ]:
import pandas as pd
import pandas_gbq
from pandas_gbq.exceptions import NotFoundException

# Example 5: to_gbq checks the destination and write mode before it builds a
# BigQuery client, so these mistakes fail fast without credentials or network.
df = pd.DataFrame({'id': [1, 2, 3], 'value': ['a', 'b', 'c']})

try:
    pandas_gbq.to_gbq(df, 'table_without_dataset', project_id='example-project')
except NotFoundException as e:
    print(f"Missing dataset in the table name: {e}")
else:
    raise AssertionError("to_gbq accepted a table name without a dataset")

try:
    pandas_gbq.to_gbq(df, 'dataset.table', project_id='example-project', if_exists='bogus')
except ValueError as e:
    print(f"Invalid if_exists: {e}")
    assert 'if_exists' in str(e)
else:
    raise AssertionError("to_gbq accepted an invalid if_exists value")

print("to_gbq argument validation completed")